# Proyecto ML — Predicción de Generación Eólica

**Memoria limpia del proyecto** · Resumen ejecutivo de todo el flujo de trabajo.

---

## Índice
1. [Descripción del problema](#1-descripción)
2. [Fuentes de datos](#2-fuentes)
3. [EDA y preprocesado](#3-eda)
4. [Ingeniería de características](#4-features)
5. [Modelos entrenados](#5-modelos)
6. [Modelo en producción](#6-produccion)
7. [Métricas finales](#7-metricas)
8. [Arquitectura del sistema](#8-arquitectura)

## 1 · Descripción del problema

El objetivo es predecir la **generación eólica semanal** (MWh) en España continental,
utilizando datos históricos de REE y variables meteorológicas de AEMET.

- **Target**: `eolica` (MWh/semana, fuente REE)
- **Horizonte**: predicción a 1 semana vista
- **Período de entrenamiento**: Enero 2022 – ≈50% split de ≈229 semanas

## 2 · Fuentes de datos

| Fuente | Descripción | Frecuencia |
|--------|-------------|------------|
| REE API | Generación por tecnología (eólica, solar, hidro…) | Diario → agregado semanal |
| AEMET API | Velmedia y racha máxima en 7 municipios eólicos | Diario → agregado semanal |
| ERA5 (NetCDF) | Radiación solar superficial (Badajoz, referencia) | Horario → no usado en modelo final |

Base de datos SQLite: `src/data/ree_generacion.db`  
Tablas: `generacion`, `predictions`, `aemet_diario`

## 3 · EDA y preprocesado

Ver notebook completo: `src/notebooks/api.ipynb`

Puntos clave:
- Correlación velmedia ↔ eólica: **0.863** · racha ↔ eólica: **0.501**
- Sin valores nulos tras limpieza; serie continua desde ene-2022
- Variables meteorológicas en escala z-score para entrenamiento de modelos
- Detección de outliers con IQR — sin eliminación (valores físicamente válidos)

## 4 · Ingeniería de características

```python
FEATURE_COLS = [
    "velmedia", "racha",          # meteorología
    "mes", "dia_semana", "dia_año",  # temporalidad
    "eolica_lag1", "eolica_lag2", "eolica_lag3", "eolica_lag7",  # lags
    "vel_ma3", "vel_ma7", "vel_ma14",    # medias móviles viento
    "racha_ma3", "racha_ma7", "racha_ma14",
    "eolica_ma7",                          # media móvil target
]
```

## 5 · Modelos entrenados

| Modelo | RMSE (MWh) | R² | MAPE |
|--------|-----------|-----|------|
| LinearRegression | ≈ 13 500 | ≈ 0.92 | 41.61 |
| Ridge | — | — | — |
| Lasso | — | — | — |
| RandomForest | — | — | — |
| LightGBM | — | — | — |
| MLPRegressor | — | — | — |
| SARIMA | — | — | — |
| SARIMAX | — | — | — |

> Completa esta tabla con los valores de `df_resultados` del notebook.

In [ ]:
# Cargar y mostrar tabla de resultados completa
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(''), '..', 'utils'))

import sqlite3, pandas as pd

# Ajusta el path según desde dónde ejecutes este notebook
DB_PATH = os.path.join(os.path.dirname(''), '..', 'data', 'ree_generacion.db')
print("DB_PATH:", os.path.abspath(DB_PATH))

## 6 · Modelo en producción

- **Modelo elegido**: `LinearRegression` (mejor equilibrio RMSE/interpretabilidad)
- **Ruta en proyecto**: `src/model/production/lr_eolica.joblib`
- **Ruta en backend (Render)**: `modelos/lr_eolica.joblib` (disco persistente)
- **Pipeline**: `{"modelo": sklearn.LinearRegression, "scaler": StandardScaler, "features": [...]}`

```python
import joblib
pipeline = joblib.load('src/model/production/lr_eolica.joblib')
modelo  = pipeline['modelo']
scaler  = pipeline['scaler']
features = pipeline['features']
```

In [ ]:
import joblib, os

MODEL_PATH = os.path.join(os.path.dirname(''), '..', 'model', 'production', 'lr_eolica.joblib')
pipeline = joblib.load(MODEL_PATH)
print("Features del modelo:", pipeline['features'])
print("Coeficientes:", dict(zip(pipeline['features'], pipeline['modelo'].coef_)))

## 7 · Métricas finales

Evalúa el modelo de producción sobre el conjunto de test:

In [ ]:
# Reproducir evaluación del modelo de producción
# (requiere que el notebook src/notebooks/api.ipynb haya ejecutado previamente
#  y los datos estén en la DB o en CSVs en src/data/)

import numpy as np
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error

# Sustituir con los arrays reales de y_test y pred_lr del notebook principal
# y_test = ...
# pred_lr = pipeline['modelo'].predict(scaler.transform(X_test))

# rmse = np.sqrt(mean_squared_error(y_test, pred_lr))
# r2   = r2_score(y_test, pred_lr)
# mape = mean_absolute_percentage_error(y_test, pred_lr)
# print(f'RMSE: {rmse:,.0f} MWh  |  R²: {r2:.4f}  |  MAPE: {mape*100:.2f}%')

## 8 · Arquitectura del sistema

```
Proyecto-ML/
├── backend/          FastAPI · desplegado en Render
│   └── app/
│       ├── api/      endpoints (charts, predictions, ree, aemet)
│       └── services/ model_loader, db_ree, aemet_client
├── frontend/         Astro.js + Tailwind · desplegado en Vercel/Netlify
│   └── src/
│       ├── pages/    index, proceso, resultados
│       └── components/charts/  Chart.js (CDN, is:inline)
├── modelos/          Modelo joblib (prod, backend)
└── src/              Código de investigación / desarrollo
    ├── notebooks/    api.ipynb (notebook principal)
    ├── model/production/  lr_eolica.joblib (copia canónica)
    ├── data/         exports, era5.nc, ree_generacion.db
    ├── utils/        fix_cell.py y otros auxiliares
    ├── resources/img/ imágenes del proyecto
    └── memoria.ipynb (este notebook)
```

**API Backend**: `https://api-ml-7var.onrender.com`  
**Endpoint predicción**: `GET /api/prediction-chart`  
**Modelo AEMET**: convierte km/h → m/s automáticamente (`vel / 3.6`)